In [10]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

Separating the rating in 3 major categories- Adults, Teens and Kids :-

In [13]:
df = pd.read_csv('data/netflix_titles_cleaned.csv')

rating_map = {
    'TV-Y': 'Kids',  'TV-Y7': 'Kids', 'TV-G': 'Kids', 'G': 'Kids', 'PG': 'Kids',
    'TV-PG': 'Teen', 'PG-13': 'Teen', 'TV-14': 'Teen',
    'TV-MA': 'Adult', 'R': 'Adult', 'NR': 'Adult', 'UR': 'Adult'
}

df['rating_category'] = df['rating'].map(rating_map)
df = df[df['rating_category'].notna()]  # drop any unmapped

print(df['rating_category'].value_counts())

rating_category
Adult    4092
Teen     3513
Kids     1189
Name: count, dtype: int64


//Next, Encode genres using MultiLabelBinarizer

In [14]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
df['genre_list'] = df['listed_in'].str.split(',').apply(lambda x: [g.strip() for g in x])

genre_dummies = pd.DataFrame(
    mlb.fit_transform(df['genre_list']),
    columns=['genre_' + g for g in mlb.classes_],
    index=df.index
)

In [17]:
# Movie=1, TV Show=0
df['is_movie'] = (df['type'] == 'Movie').astype(int)

# Top 10 countries as dummies, rest = 'Other'
top_countries = df['country'].value_counts().head(10).index
df['country_clean'] = df['country'].apply(lambda x: x if x in top_countries else 'Other')
country_dummies = pd.get_dummies(df['country_clean'], prefix='country')

# Decade
df['release_decade'] = (df['release_year'] // 10) * 10

In [18]:
features = pd.concat([
    df[['is_movie', 'release_decade', 'duration_minutes']].fillna(0),
    genre_dummies,
    country_dummies
], axis=1)

target = df['rating_category']

# Save for Phase 3
features.to_csv('data/features.csv', index=False)
target.to_csv('data/target.csv', index=False)

print("Features shape:", features.shape)

Features shape: (8794, 56)
